# CSC 391 Mini-Project 1: Spamlord

This assignment is your chance to become a __Dark Lord__ of spam email!
Yes, you too can build regular expressions (`RegExes`) to spread evil throughout the galaxy. 
The goal of this assignment is to use `RegExes` to extract phone numbers and
email addresses from documents found on the web.
This may seem easy at first, as you can write very simple `RegExes` to catch similar cases such as `336-758-1234` or `deac1@wfu.edu`.
On the other hand there are various different ways people write their emails in `HTML` documents, some to prevent scrapers from capturing them easily, which you will learn in more detail in the upcoming sections.
If you really were a malicious actor, you could then use these extracted addresses to bombard unsuspecting victims with spam!
Of course, we would never do anything nefarious like that in `CSC391`. 
Instead our goal will be to work with raw data and gain some experience with `RegExes`.

<a id="contents"></a>
## Contents

Listed below are the contents of mini-project 1. In the `Data Exploration` section, you will look into the dataset we will use in this pilot study. In the `Example Approach` section, you will learn more about the specifics of our email address and phone number catching task, and see an example implementation. In the `Evaluation` section, you will learn how to evaluate `RegExes` on our dataset. `Cases to Consider` section provides you with tips on the tricky cases you may run into. `Your Approach` section is the place where you actually do the coding. In the `Reflection` section, you will answer questions on the environmental and societal impacts of email spamming. Please read through the notebook before you start working through the assignment.

* [`Part 1. Data Exploration`](#data_exploration)
* [`Part 2. Example Approach`](#example_approach)
* [`Part 3. Evaluation`](#evaluation)
* [`Part 4. Cases to Consider`](#cases_to_consider)
* [`Part 5. Your Approach`](#your_approach)
* [`Part 6. Reflection`](#reflection)



<a id="roadmap"></a>
## Tasks

As an overview, there are`4` functions you need to implement in this assignment:
* In `Part 5. Your Approach`: **[`find_phone_numbers()`](#find_phone_numbers)** and **[`find_emails()`](#find_phone_emails)**
* In `Part 6. Reflection`: **[`carbon_dioxide_emissions()`](#carbon_dioxide_emissions)** and **[`government_response()`](government_response)**

You will write your `RegExes` in **`find_phone_numbers()`** and **`find_phone_emails()`**, which make up the meat of this assignment and will take the longest. A very short implementation is needed for the **`carbon_dioxide_emissions()`** function. You will provide a short answer to an open-ended question in the **`government_response()`** function.

<a id="setup"></a>
## Part 0. Setup

**Check data files.** The cell below checks for the existance of data folder. 

In [1]:
%%bash

# Check if the ./data folder exists.
# Download it if not found.
if [[ ! -d "../data" ]]
then
    echo "Missing extra files. Download archive from Canvas..."
fi

**Import Modules.** Run the next cell to import the necessary modules used in this assignment.

In [23]:
""" Modules included in the Python Standard Library """

# We use features from io and os modules for opening files and writing to them
from io import open
import os

# re module contain methods for using regular expressions
import re

# typing module contains type objects. We will use these types to ensure that 
# the inputs and outputs passed to the functions you will be implementing are 
# of the correct type
from typing import List

**Import Custom Modules.** Run the next cell to import the custom functions and classes used in this assignment.

In [24]:
""" Custom functions and classes """

# Helper functions we will use later
from util import process_dir, get_gold, score

**Note:** **DO NOT** import and use any other packages outside of the Python standard
library.

<a id="data_exploration"></a>
## Part 1. Data Exploration

Let's start by taking a look at what our data actually looks like.
This should always be one of the first things you do whenever you are solving a problem that requires working with data.

**Development Set.** In order to make your life easier on this and future homeworks, we will be
giving you some data to study and test your code on, which we call a
`development set` or a `dev set`.
Using a dev set to test and evaluate your methods is an extremely common approach in `Natural Language Processing`.
More generally, coming up with a robust set of test cases to evaluate your work against is an extremely important part of writing good code.

Our dev set consists of a bunch of `HTML` documents (the personal
homepages of some `WFU CS` professors) that we have scraped from the web and downloaded for you. 
If you are not familiar with the details of `HTML` or its syntax, it's fine. 
For the purposes of this assignment, all you need to know is that the inputs are text files (with some formatting) that contain the (possibly obfuscated) emails and phone numbers that we want to extract.
You can find all of these `HTML` documents in the `data/dev` directory.

**Exploration.** To visualize the documents in our dev set, you can take any of the files and open them in a browser of your choice!
For example, by right-clicking on `data/dev/fulp.html` and clicking `open with -> Firefox`.
You can also try double-clicking, which usually opens the page in your default browser.
Feel free to change the filename to some of the other files in `data/dev`
(i.e. `santago` to `ballard`, `burg`, etc.) to take a look at some of the other
faculty pages.
```
Mini Task: Open data/dev/santago.html and look for email addresses/phone numbers.
What kind of regular expressions you would need to catch these?
```
You should find that, as expected, the page you opened looks exactly like a
faculty webpage, possibly minus some images which we didn't download along
with the `HTML`, but that's fine, as we are only interested in the text.
As is common for faculty pages, these pages have contact information like email
addresses and phone numbers. Our goal is to write regular expressions
that we can use to automatically match and extract these from the webpages.

**Reading the HTML Documents.**
We have seen what the files look like as webpages.
However, we are interested in the text contents, as that is what we will be matching with our regular expressions.
Let's try reading in the `data/dev/alqahtani.html` as a single giant text string.

In [13]:
# Open and read a file as a gigantic string
with open("../data/dev/alqahtani.html", 'r', encoding='ISO-8859-1') as file:
    full_text = file.read()
    print(full_text)

<!DOCTYPE html>
<html lang="en"><head>
<meta http-equiv="content-type" content="text/html; charset=UTF-8">
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<link rel="profile" href="http://gmpg.org/xfn/11">
<title>Sarra Alqahtani</title>
<meta name="robots" content="max-image-preview:large">
<link rel="dns-prefetch" href="https://fonts.googleapis.com/">
<link rel="dns-prefetch" href="https://s.w.org/">
<link rel="alternate" type="application/rss+xml" title="Sarra Alqahtani Â» Feed" href="https://alqahtas.sites.wfu.edu/feed/">
<link rel="alternate" type="application/rss+xml" title="Sarra Alqahtani Â» Comments Feed" href="https://alqahtas.sites.wfu.edu/comments/feed/">
<script type="text/javascript">
window._wpemojiSettings = {"baseUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/72x72\/","ext":".png","svgUrl":"https:\/\/s.w.org\/images\/core\/emoji\/13.1.0\/svg\/","svgExt":".svg","source":{"concatemoji":"https:\/\/alqahtas.sites.wfu.edu\

Okay, it's a bit long and hard to parse, but it seems reasonable!
There's a bunch of somewhat cluttered `HTML` markup, but it's all in text form and if we search through it, it looks like all of the text from the page including the emails and phone numbers, is somewhere in there.
In the remainder of this assignment, you will come back to printing the strings for the `HTML` documents to understand the cases to improve your regular expressions by finding the cases that they are missing.

<a id="example_approach"></a>
## Part 2. Example Approach

Now that we have learned how to load `HTML` documents from our dataset and inspect them, let's see if we can extract some email addresses using a regular expression pattern.
In this case we will use a super-simple pattern that just looks for 1 or more alphanumeric characters or periods followed by an `@` followed by 1 or more
alphanumeric characters or periods, followed by `.edu`. This is just the usual
format of an email address.
```
deac1@wfu.edu
```
We can achieve our goal with the following regular expression.
```
([\w\.]+)@([\w\.]+\.edu)
```
Let's break down what this pattern does!
* `\w`: A word character, same as the regular expression `[A-Za-z0-9_]`. Note that it matches `_` too!
* `[\w.]`: Matches any word character or `.`. Note that we don't need to use an escape character before `.`, since any character other than `^`, `-`, `\` or `]` is interpreted as a literal in a character class (which is denoted by `[]`).
* `[\w\.]+`: Matches at least 1 word character or period. This is the pattern we wanted to match for the name part of the regular expression, so we are done!
* `[\w\.]+\.edu`: Matches at least 1 word character or period followed by `.edu`. Note that we have to use an escape character before the period this time around.
* `(...)`: Capture groups for saving the matches.
That is, the regular expression engine will not only match the expression inside `()` to a part of an expression, but also record the matched part in a capture group, which we can retrieve later.
* `([\w\.]+)` and `([\w\.]+.edu)`: The capture groups are used to capture the part of the email before and after the `@`, respectively.

Let's test our pattern on a short string.
Notice how we use the formatter character `%` in combination with tuples to build strings in our desired format.


In [3]:
"""
The function re.findall takes a regex pattern and a text string and returns all 
matches in the string as a list. Each match in the list is a tuple of the 
capture groups in the expression. So in this case, each element in matches will 
be a tuple of form:

    (stuff before '@', stuff after '@')

"""
# Create the example string and patter
example = 'The email address is deac1@wfu.edu.'
simple_pattern = '([\w\.]+)@([\w\.]+\.edu)'

# Find the matches, which are returned as a list containing two-tuples
matches = re.findall(simple_pattern, example)


# Iterate over the matches 
for m in matches:
    # Print matches
    print("The first capture group is: %s" % m[0])
    print("The second capture group is: %s" % m[1])
    print("The first and second capture groups are: %s and %s" % m)

    # Put the email back together
    email = '%s@%s' % m
    print(email)

The first capture group is: deac1
The second capture group is: wfu.edu
The first and second capture groups are: deac1 and wfu.edu
deac1@wfu.edu


Observe how we put the email back together using capture groups.
We can now wrap the same code above in a function, that takes in a sring and returns the list of emails found in the string.

In [15]:
# Define our function
def example_find_emails(full_text: str) -> List[str]:
    """
    This is an example function that takes a string and finds the emails in
    it. Returns the found emails in a list of strings. The returned emails
    must follow the canonical format:

              'someone@something'

    We use -> to show the return type of the function. Typing isn't explicitly 
    enforced in Python, so we didn't have to specify the return type of our 
    function, but we are specifying them in this assignment to help you tackle
    errors in an easier way.

    full_text (str): Full text of the html file read.
    """
    # The simple pattern
    simple_pattern = '([\w\.]+)@([\w\.]+\.edu)'
    matches = re.findall(simple_pattern, full_text)

    # Iterate over the matches
    res = []
    for m in matches:
        email = '%s@%s' % m
        res.append(email)
    return res

Let's see if our function works as expected.

In [16]:
# Call our function
example_line = 'The email address is deac1@wfu.edu.'
example_find_emails(example_line)

['deac1@wfu.edu']

Great! We now have a simple function that we can call on a string to extract
email in simple forms.
Since we also need to extract phone numbers, we need a function that returns a list of phone number strings.
To do so, we need to find a regular expression string that matches phone numbers in various different forms and implement our function similar to the **`example_find_emails()`** function.
For demonstration, we are sharing a dummy function that just returns an empty list.
When you are implementing this section in `Part 4. Your Turn` you will need to extract the phone numbers using regular expressions.

In [17]:
def example_find_phone_numbers(full_text: str) -> List[str]:
    """
    This is an example function that takes in a string and finds the phone 
    numbers in it. Returns the found numbers in a list of strings. The returned
    numbers must follow the canonical format, where # represent digits:

              '###-###-#####'

    full_text (str): Full text of the html file read.
    """
    return []

Let's try calling our example phone number function.

In [18]:
example_line = 'The phone number is 336-758-6000.'
example_find_phone_numbers(example_line)

[]

Our function returned an empty list, which is what we expected!

<a id="evaluation"></a>
## Part 3. Evaluation

Evaluation is a crucial step of any kind of `NLP` or `ML` project.
For us to be able to evaluate how our functions are doing, we need some sort of grounding.
In addition to the `HTML` documents, the `data` directory also contains
another file, `data/devGOLD`.
You can think of this file as the answer key corresponding to the documents in `data/dev`.
It contains all the correctly extracted phone numbers and emails from all the documents in `data/dev`, in a particular format so your scripts as well as our grading scripts can read them easily.

### Part 3.1. Format Matches


Each line in the `data/devGOLD` file represents one extracted email address
or phone number in the form of a 3-tuple. 
Each tuple is represented as 3 strings separated by vertical bars ("|"). 
You can open the `data/devGOLD` file to see for yourself.

```
alqahtani|e|sarra-alqahtani@wfu.edu
```

* The **first** string is the name of the file that the match came from where the `.html` extension removed.
* The **second** string is an `e` if the match was an email address, or a
`p` if it was a phone number.
* The **third** string is the actual extracted email address or phone number itself,
in the following canonical form.

```
  user@example.com
  336-758-1234
```

To sum up, the answers in the ```data/devGOLD``` file and the outputs
generated by your implementation should take the form of `Python` tuples
that look like the following.

```
  (filename, match type, match value)
```

The functions we have coded so far can take in a string and return the list of extracted email addresses and phone numbers in a their respective list.
To be able to evaluate our functions, we need another function that will call our functions on each line of a file, and output the results in the specified format above. The function shared in the next cell, **`example_process_file()`** does exactly this.

In [19]:
def example_process_file(filename: str, data_directory: str):
    """
    Function we wrote to call the functions listed below on each line of a file 
    with the given filename. It returns a list of 3-tuples representinting the 
    found matches in the specified evaluation format.
    
    * example_find_emails()
    * example_find_phone_numbers functions 

    """
    # The format of our evaluation matches requires stripping the ".html" 
    # extension from our filenames.
    filename_no_ext, ext = filename.split('.')
    absolute_file_path = os.path.join(data_directory, filename)
    res = []
    with open(absolute_file_path, 'r', encoding='ISO-8859-1') as file:
        # Read the full text
        full_text = file.read()
        # Call example_find_emails
        emails = [(filename_no_ext, 'e', e) for e in example_find_emails(full_text)]
        # Call example_find_phone_numbers
        phone_numbers = [(filename_no_ext, 'p', p) for p in example_find_phone_numbers(full_text)]
        # Add the newly extracted emails and phone numbers to our list
        res += emails + phone_numbers

    return res

Let's check which emails and phone numbers our functions will find in a given file.
Note that we do not expect to get any phone numbers since the `example_find_phone_number` is set to return an empty list.

In [20]:
result = example_process_file('alqahtani.html', '../data/dev')
print(result)

[('alqahtani', 'e', 'alqahtani@wfu.edu')]


Success! It looks like we got our first match! The output of our function in
this case is a list of matches, where each match is a tuple in the following format. The reason we want our output in this format is so that it works
with our automated scoring later.
```
(file, e or p indicating email or phone number, extracted email or phone number)
```

In this case we can see that we have just a single match from the file `data/dev/alqahtani.html` which is an email and is the address `alqahtani@wfu.edu`.
Note that we only searched in this one file!

So far we have seen how we can process a single file.
However, our dev set consists of many such files.
We need a function to loop over all of them, process them, and return all the extracted addresses. 
This function is provided for you in `util.py`, and it is named **`process_dir()`**!
You shouldn't modify this function, but we encourage you to take a look at how it is implemented.

In [95]:
all_results = process_dir('../data/dev/', example_process_file)

print(all_results)

[('plemmons', 'e', 'plemmons@wfu.edu'), ('plemmons', 'e', 'plemmons@wfu.edu'), ('chen', 'e', 'chenm@wfu.edu'), ('alqahtani', 'e', 'alqahtani@wfu.edu'), ('luthy', 'e', 'luthyka@wfu.edu'), ('luthy', 'e', 'engineer@wfu.edu'), ('luthy', 'e', 'engineer@wfu.edu'), ('vanbastelaer', 'e', 'vanbasm@wfu.edu'), ('vanbastelaer', 'e', 'vanbasm@wfu.edu'), ('canas', 'e', 'canas@wfu.edu'), ('burg', 'e', 'burg@wfu.edu'), ('pease', 'e', 'peasejb@wfu.edu'), ('pease', 'e', 'peasejb@wfu.edu'), ('pease', 'e', 'biology@wfu.edu'), ('cho', 'e', 'choss@wfu.edu'), ('cho', 'e', 'choss@wfu.edu')]


Looks like we got quite a few more matches, even with our very simple pattern. You may have also noticed that our results have quite a few duplicates. If you
examine the corresponding files, you can see that this is happening because
the same email address (or phone number) appears more than once in the file. Don't worry about this for now, we wil be careful to strip out duplicates later when we are doing our scoring.

### Part 3.2. Compare to Gold

The final step of the evaluation process is straightforward: all that needs to be done is to load the correct answers for the dev set from the provided file (`data/devGOLD`) and compare them to the matches that were generated by our function. 
We provide this helper function in `util.py`: it is called **`get_gold()`**.
Again, you shouldn't modify it, but you can take a peek at it if you are curious what it's doing. Let's use it to read the gold (correct) matches from the provided file.

In [21]:
all_gold_matches = get_gold('../data/devGOLD')
print(all_gold_matches)

[('alqahtani', 'e', 'sarra-alqahtani@wfu.edu'), ('alqahtani', 'p', '336-758-3129'), ('ballard', 'e', 'ballard@wfu.edu'), ('ballard', 'p', '336-758-4137'), ('burg', 'e', 'burg@wfu.edu'), ('burg', 'p', '336-758-4465'), ('canas', 'e', 'canas@wfu.edu'), ('canas', 'p', '336-758-5355'), ('chen', 'e', 'chenm@wfu.edu'), ('chen', 'p', '336-758-3732'), ('cho', 'e', 'choss@wfu.edu'), ('devaraconda', 'e', 'aditya@wfu.edu'), ('fulp', 'e', 'fulp@wfu.edu'), ('fulp', 'p', '336-758-3752'), ('khuri', 'e', 'natalia.khuri@wfu.edu'), ('luthy', 'e', 'luthyka@wfu.edu'), ('luthy', 'p', '336-702-1964'), ('pauca', 'e', 'paucavp@wfu.edu'), ('pauca', 'p', '336-758-5454'), ('pease', 'e', 'peasejb@wfu.edu'), ('pease', 'p', '336-758-5567'), ('plemmons', 'e', 'plemmons@wfu.edu'), ('plemmons', 'p', '336-391-2495'), ('santago', 'e', 'ps@wfu.edu'), ('santago', 'p', '336-758-4190'), ('santago', 'p', '336-758-4982'), ('vanbastelaer', 'e', 'vanbasm@wfu.edu')]


As expected, these exactly match the output format that we showed earlier for the file processing function.
This will make it easy to compare our answers to the gold answers, for which we provide a helper function in `util.py`, called **`score()`**.
You are welcome to take a look if you are curious. 
It takes in a list of your predicted matches, the output of the function you will write, and a list of correct/gold matches, read from the `data/devGOLD` file.
It compares the two and calculates how they overlap, printing out a bunch of information. You can interpret the results as follows:

* **`The true positive`** section displays emails and phone numbers which are in
both your list of matches and the gold matches list.
These are examples that your regular expressions correctly found.

* **`The false positive`** section displays matches which your regular expressions extracted but which are not in the gold matches list.
These are incorrect and show where your method may have been too
broad/aggressive.

* **`The false negative`** section displays emails and phone numbers which your code did not match, but which do exist in the html files.
These are the matches your code missed.

Your goal, then, is to reduce the number of false positives and false negatives
to 0.
At the bottom of the output you can see the total counts of `true positives`, `false positives`, and `false negatives`.

Let's try evaluating our existing super-basic method using the method described above.

In [22]:
guess_list = process_dir('../data/dev', example_process_file)
gold_list = get_gold('../data/devGOLD')
score(guess_list, gold_list)

Guesses (11): 
{('alqahtani', 'e', 'alqahtani@wfu.edu'),
 ('burg', 'e', 'burg@wfu.edu'),
 ('canas', 'e', 'canas@wfu.edu'),
 ('chen', 'e', 'chenm@wfu.edu'),
 ('cho', 'e', 'choss@wfu.edu'),
 ('luthy', 'e', 'engineer@wfu.edu'),
 ('luthy', 'e', 'luthyka@wfu.edu'),
 ('pease', 'e', 'biology@wfu.edu'),
 ('pease', 'e', 'peasejb@wfu.edu'),
 ('plemmons', 'e', 'plemmons@wfu.edu'),
 ('vanbastelaer', 'e', 'vanbasm@wfu.edu')}
Gold (27): 
{('alqahtani', 'e', 'sarra-alqahtani@wfu.edu'),
 ('alqahtani', 'p', '336-758-3129'),
 ('ballard', 'e', 'ballard@wfu.edu'),
 ('ballard', 'p', '336-758-4137'),
 ('burg', 'e', 'burg@wfu.edu'),
 ('burg', 'p', '336-758-4465'),
 ('canas', 'e', 'canas@wfu.edu'),
 ('canas', 'p', '336-758-5355'),
 ('chen', 'e', 'chenm@wfu.edu'),
 ('chen', 'p', '336-758-3732'),
 ('cho', 'e', 'choss@wfu.edu'),
 ('devaraconda', 'e', 'aditya@wfu.edu'),
 ('fulp', 'e', 'fulp@wfu.edu'),
 ('fulp', 'p', '336-758-3752'),
 ('khuri', 'e', 'natalia.khuri@wfu.edu'),
 ('luthy', 'e', 'luthyka@wfu.edu'),
 ('

Looks reasonable! It appears that our basic method produced 30 matches.
There were: 
* 8 `true positives`, which are matches that we found that were in the gold set;
* 3 `false positives`, which are matches that we found that were NOT in the gold set;
* 19 `false negatives`, which are matches in the gold set that we did NOT find.

This seems like a pretty good start, but there are still work to be done. Figuring out how to extract matches without extracting non-email text is up to you!


### Part 3.3. Grading Rubric

Your grade will consist of three parts, totalling 100 points.

* The **first**, worth 50 points, scores how well your implementation does on the
development set.
For these examples you're given the correct answers, so you should aim to get 100% of them correct!

* The **second** part of your grade, worths 30 points, will be based on how well your
regular expressions find emails and phone numbers in a different set of
examples, the `test set`. 
This test set is hidden and only the instructor knows what is in it!
Because you don't know exactly what trickery goes on in this test set, you should be creative in thinking of different ways of writing (and hiding) emails and phone numbers.

You are not expected to perform perfectly on the test set as you don't know
what is in it, or have the correct answers (just like in real life).
As long as you manage to achieve some reasonable performance (compared to a benchmark that we provide), you will get full points! 
The benchmark is set at **65** test errors or fewer.

* The **third** part is a section worth 20 points designed to get you thinking about ethical issues surrounding spam emails.

Here are the equations we use to calculate the scores for the two parts, where
`e` is the total number of errors (`false negatives` and `false positives`) for
each part:

__Dev:__

```
  if e < 10 then score(e) = 50 - e
  else if e >= 10 then score(e) = 40
```

__Test:__

```
  if e <= 65      then score(e) = 30
  else if 65 < e  then score(e) = 30 - (e - 65) * 0.2
```

__Note:__ This sort of two-stage evaluation (a known development set and a
hidden test set) is a very commonly used approach in machine learning!
Evaluating on a development set where we have the "right" answers lets us
measure our performance precisely and improve our approach, while a test set
that is hidden from us until later allows us to see how we perform
"out in the wild", on examples that we might not have been able to tailor
our methods to.

<a id="cases_to_consider"></a>
## Part 4. Cases to Consider

As you implement your regular expressions and analyze the `HTML` files that your approach isn't getting quite right, you will develop an understanding of which cases to consider.
Your development workflow will be as follows:
* You will start with a simple regular expression.
* You will evaluate your simple approach against the gold matches.
* You will find the files for which your approach is failing and try to identify how you can improve your regular expression to do better.
* You will go back to evaluation step and repeat until you are satisfied.

This is how a real life `NLP` or `ML` practioner would approach an unknown task!
For our assignment, to make things a little more concrete, we are providing you with some examples to illustrate exactly what your implementation should be able to do if it's working correctly.
The list we provide here is not comprehensive, so you may find out about cases that we haven't covered here.

### Part 4.1. Extracting Phone Numbers

Your program should be able to process text that looks like the examples shown
on the left, and extract the phone number on the right. Note that we want
the result in a standardized form.

```
# Various formats
Phone:  (336) 758-0293 =>  336-758-0293
Tel (+1): 336-758-0293 =>  336-758-0293

# HTML Markup
<a href="contact.html">TEL</a> +1&thinsp;336&thinsp;758&thinsp;0293 => 336-758-0293
```
Here are some notes:

* The last line shows an example method people use to make hide their phone numbers using `HTML` markup.
You are required to cover such cases.

* You can assume we are only working with North American phone numbers,
so all numbers will be of the form: `[3 digit area code]-[3 digits]-[4 digits]`.

* Your solution may also extract fax numbers, which look exactly like
phone numbers.
This is fine, you are not expected to distinguish between the two.

### Part 4.2. Extracting Email Addresses

Similarly to the phone numbers, we are also interested in processing text
containing (possibly obfuscated) email addresses and returning the corresponding
email addresses in a standard form.

```
# Ordinary email addresses
deac1@wfu.edu => deac1@wfu.edu

# Hidden email addresses
deac1 WHERE wfu DOM edu => deac1@wfu.edu
deac1(at)wfu.edu => deac1@wfu.edu
deac1 at cs dot wfu dot edu => deac1@cs.wfu.edu
```
Below are some notes/questions to guide you. Make sure to account for different cases (lowercase, uppercase, mixed) for each of the following points!
* Notice the different ways people write the `@` sign. 
  Can you identify a few?
* What about the alternative ways of writing `.` in emails?
  Make sure to account for different cases (lowercase, uppercase, mixed)!
* What are some popular top level domain names?
  To get full credit on the assignment, it is sufficient to consider `com`, `gov`, `org`, `edu`, `info`.
  Remember to account for cases!
* Are there other ways people write their emails in plain english?
  What are some of the common ones?

### Part 4.3. Cases to not Worry About

Although you should aim to make your regexes as powerful and general-purpose as
you possibly can, there are some cases that are difficult or impossible to
handle with regexes and which we don't expect you to be able to deal with.

These include:

* Anything involving images or other non-text ways of displaying emails or phone
numbers.
* Examples that require parsing names into parts, like:.

```
"alba torres"@wfu.edu
```

* Particularly clever/difficult examples that don't contain much or any
part of the actual email address. For example,

```
To send me email, try the simplest address that makes sense.
```

<a id="your_approach"></a>
## Part 5. Your Approach

The example functions we shared so far only allows us to retrieve a subset of the present emails and none of the phones in our dataset.
In this section, you will implement your version of the example functions, and test your implementations 
Your task is to modify the functions **`find_emails()`** and **`find_phone_numbers()`** given below.
We provide you with a placeholder code, but you will modify them.
Here we share some notes/tips that may be helpful in your implementations:
* We recommend starting with the phone numbers as the number of cases to be considered are significantly smaller.
* You can use separate regular expressions for separate cases, and combine your results into a list before returning.
This will make writing regular expressions easier.
* You may get long regular expressions as you try to cover each email case.
Don't get discouraged and make use of `|`.
* Although they are mostly the same, different regular expression engines differ in subtle ways, especially true for the way escape characters etc. are interpreted.
If you are using an external website to test your `RegExes`, be aware that your `RegExes` may not work out of the box when you move them over to `Python` due to this distinction.

In [25]:
# TODO: Implement your approach here!
def find_phone_numbers(full_text: str) -> List[str]:
    """
    Takes in a line from an html document as a string and finds the phone 
    numbers in it. Returns the found numbers in a list of strings. The returned
    numbers must follow the canonical format, where # represent digits:

              '###-###-#####'

    NOTE: DO NOT CHANGE THIS INTERFACE, as it will be called directly by
    the submit script.

    full_text (str): Full text of the html file read.
    """
    # CODE START
    pattern = r'(?<![<>(\w\.\-:\s][fF][aA][xX][)\w\.\-:\s])[\.\s*\-\_\>\<\;][(]?([1-9][\d]{2})[)]?[\-\.\s*]{1,3}([\d]{3})[\-\.\s*]{1,3}([\d]{4})(?![\w\.\-:\s<>][(]?[[fF][aA][xX][)]?)'
    # Iterate over the matches
    
    res = []
    matches = re.findall(pattern, full_text)
    for m in matches:
        phone = '%s-%s-%s' % m
        res.append(phone)
    
    
    return res
    # CODE END

In [26]:
# TODO: Implement your approach here!
def find_emails(full_text: str) -> List[str]:
    """
    Takes in a line from an html document as a string and finds the emails in
    it. Returns the found emails in a list of strings. The returned email
    must follow the canonical format:

              'someone@something'

    NOTE: DO NOT CHANGE THIS INTERFACE, as it will be called directly by
    the submit script.

    full_text (str): Full text of the html file read.
    """
    # CODE START
    patterns = [r'([\w\.\-]+)\s*(?:@|[\"\'(\s]at[\"\'\s)]|AT)\s*([^(?:email)][\w\.\-]+)\s*(?:\.|DOT|dot)\s*(edu|com|gov|org|info)',
               r'<(?:.*?)>NOSPAM<(?:.*?)>\s*([\w\.\-]+)\s*(?:(?:<(?:.*?)>NOSPAM<(?:.*?)>\s*)([\w\.\-]+)\s*)?(?:(?:<(?:.*?)>NOSPAM<(?:.*?)>\s*)([\w\.\-]+)\s*)?(?:(?:<(?:.*?)>NOSPAM<(?:.*?)>\s*)([\w\.\-]+)\s*)?<(?:.*?)>NOSPAM<(?:.*?)>\s*(?:@|[\"\'(\s]at[\"\'\s)]|AT)\s*<(?:.*?)>NOSPAM<(?:.*?)>\s*([\w\.\-]+)\s*<(?:.*?)>NOSPAM<(?:.*?)>\s*(?:\.|DOT|dot)\s*<(?:.*?)>NOSPAM<(?:.*?)>\s*(edu|com|gov|org|info)',
               r'<(?:.*?)>([\w\.\-]+)<(?:.*?)>\s*(?:\'?(?:@|[aA][tT])\'?)\s*([\w\.\-]+)\s*(?:\.|DOT|dot)\s*(edu)']

    res = []
    for pattern in patterns:
        matches = re.findall(pattern, full_text)
        # Iterate over the matches
        for m in matches:
            if(len(m) > 3):
                m = list(m)
                m[0:len(m)-2] = [''.join(m[0:len(m)-2])]
                m = tuple(m)
            email = '%s@%s.%s' % m
            res.append(email)
            
    
    return res
    # CODE END

In [27]:
# DO NOT CHANGE
def process_file(filename: str, data_directory: str):
    """
    Function we wrote to call the functions listed below on each line of a file 
    with the given filename. It returns a list of 3-tuples representinting the 
    found matches in the specified evaluation format.
    
    * find_emails()
    * find_phone_numbers functions 

    """
    # DO NOT CHANGE
    filename_no_ext, ext = filename.split('.')
    absolute_file_path = os.path.join(data_directory, filename)
    res = []
    with open(absolute_file_path, 'r', encoding='ISO-8859-1') as file:
        # Read the full text
        full_text = file.read()
        
        # Call find_emails
        emails = [(filename_no_ext, 'e', e) for e in find_emails(full_text)]
        
        # Call find_phone_numbers
        phone_numbers = [(filename_no_ext, 'p', p) for p in find_phone_numbers(full_text)]
        
        # Add the newly extracted emails and phone numbers to our list
        res += emails + phone_numbers

    return res

Similar to the example functions, you can run your functions on all of the dev set and compare your m=found matches with the gold set matches.

In [28]:
guess_list = process_dir('../data/dev', process_file)
gold_list = get_gold('../data/devGOLD')
score(guess_list, gold_list)

Guesses (31): 
{('alqahtani', 'e', 'sarra-alqahtani@wfu.edu'),
 ('alqahtani', 'p', '336-758-3129'),
 ('ballard', 'e', 'ballard@wfu.edu'),
 ('ballard', 'p', '336-758-4137'),
 ('burg', 'e', 'burg@wfu.edu'),
 ('burg', 'p', '336-758-4465'),
 ('canas', 'e', 'canas@wfu.edu'),
 ('canas', 'p', '336-758-5355'),
 ('chen', 'e', 'chenm@wfu.edu'),
 ('chen', 'p', '336-758-3732'),
 ('cho', 'e', 'choss@wfu.edu'),
 ('devarakonda', 'e', 'aditya@wfu.edu'),
 ('fulp', 'e', 'fulp@wfu.edu'),
 ('fulp', 'p', '336-758-3752'),
 ('khuri', 'e', 'natalia.khuri@wfu.edu'),
 ('luthy', 'e', 'engineer@wfu.edu'),
 ('luthy', 'e', 'luthyka@wfu.edu'),
 ('luthy', 'p', '336-702-1964'),
 ('pauca', 'e', 'paucavp@wfu.edu'),
 ('pauca', 'p', '336-758-5454'),
 ('pease', 'e', 'biology@wfu.edu'),
 ('pease', 'e', 'peasejb@wfu.edu'),
 ('pease', 'p', '336-758-5322'),
 ('pease', 'p', '336-758-5323'),
 ('pease', 'p', '336-758-5567'),
 ('plemmons', 'e', 'plemmons@wfu.edu'),
 ('plemmons', 'p', '336-391-2495'),
 ('santago', 'e', 'ps@wfu.edu'

From the list above, select a file for which your approach is outputting an incorrect result, print the contents of this file using the next cell, and look into why your regular expression may not be capturing the missed emails/numbers.
As you make improvements to your functions, come back to this section and repeat the process and you are satisfied with the reuslts.

In [102]:
selected_file = 'erway.html'
result = process_file(selected_file, '../data/dev')
print(result)

[]


<a id="reflection"></a>
## Part 6. Reflection

<a id='carbon_dioxide_emissions'></a>
**Carbon Dioxide Emissions.**
Wait! Don’t send that spam email! Like most other tasks involving electricity, sending that little spam email actually releases carbon dioxide into the atmosphere.
Each email sent requires not only the electricity you use on your personal device, but also the energy to store and transmit the message through data centers.
According to the carbon footprint expert `Mike Berners-Lee`’s 2010 book [`How Bad are Bananas: The Carbon Footprint of Everything`](http://www.goodreads.com/book/show/7230015-how-bad-are-bananas), the average spam email has a carbon footprint equivalent to `0.3 grams` of carbon dioxide.
Furthermore a normal email has a footprint  of `4 grams` of carbon dioxide and an email with long attachments can have a carbon footprint of `50 grams` carbon dioxide.

It is estimated that globally around [120 billion spam emails](https://talosintelligence.com/reputation_center/email_rep?cid=27273&industry=agency&offset=390) are sent every day.
Calculate the global carbon emission for a day’s worth of spam emails.
Provide the answer in tons.

**Note:** Use `1 gram = 1.10231e-6 tons`.

In [12]:
# TODO: Modify this function so that it returns your solution
def carbon_dioxide_emissions():
    gram_emissions = 1.2e11 * 0.3
    carbon_emissions = gram_emissions * 1.10231e-6
    
    return carbon_emissions
carbon_dioxide_emissions()

39683.16

In [3]:
def explanation():
    explanation = "Our algorithm ended up having 5 false positives and 1 false negatives. The one false negative we have is professor Devarakonda's email. The name 'Devarakonda' is spelled wrong in the gold list so this should not count as a false negative. The 5 false postives are the department emails and phone numbers from professors' pages. It's quite complicated to further separate those out from professor's information because they are valid emails and phone numbers. We also think it's not harmful to include those department information because it expanded our spam list a little but still can count as the targeted information"
    return explanation
explanation()


"Our algorithm ended up having 5 false positives and 1 false negatives. The one false negative we have is professor Devarakonda's email. The name 'Devarakonda' is spelled wrong in the gold list so this should not count as a false negative. The 5 false postives are the department emails and phone numbers from professors' pages. It's quite complicated to further separate those out from professor's information because they are valid emails and phone numbers. We also think it's not harmful to include those department information because it expanded our spam list a little but still can count as the targeted information"